In [1]:
from pyspark.sql import SparkSession

# Build SparkSession with the Kafka SQL Connector
spark = SparkSession.builder \
    .appName("Kafka_Spark_Structured_Streaming") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0") \
    .getOrCreate()

print("Spark Version:", spark.version)
print("Spark Session with Kafka Connector created successfully!")

Spark Version: 3.3.0
Spark Session with Kafka Connector created successfully!


### Exercise 2: Connecting Spark to Kafka (Jupyter)
Out of the box, Spark doesn't have the Kafka drivers. We must configure our SparkSession to download the Kafka SQL package.

Step 1: Initialize the Streaming Spark Session
Run this cell in your Jupyter Notebook:

In [2]:
# Read the live stream from Kafka
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "live_events") \
    .option("startingOffsets", "latest") \
    .load()

### Exercise 3: Building the Streaming Pipeline (Jupyter)
Instead of spark.read (which is for batch files), we will use spark.readStream to constantly listen to the Kafka topic.

Step 1: Subscribe to the Kafka Topic

In [3]:
from pyspark.sql.functions import explode, split, col

# 1. Cast the binary Kafka 'value' to a readable String
events_df = kafka_df.selectExpr("CAST(value AS STRING) as message")

# 2. Split the message into individual words
words_df = events_df.select(explode(split(col("message"), " ")).alias("word"))

# 3. Count the occurrences of each word continuously
word_counts = words_df.groupBy("word").count()

print("Streaming transformations defined!")

Streaming transformations defined!


### Step 2: Parse and Transform the Data
Kafka sends data as raw binary bytes. We need to cast it to a string, then we will split the sentences into words and count them.

In [4]:
# Start the streaming query
streaming_query = word_counts.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("live_word_counts") \
    .start()

print("Stream is now active and listening to Kafka!")

Stream is now active and listening to Kafka!


### Step 3: Start the Stream into a Memory Sink
Since we are in a Jupyter Notebook, we will tell Spark to dump the live results into a temporary in-memory SQL table called live_word_counts so we can query it interactively.

In [7]:
# Query the live aggregating table
spark.sql("SELECT * FROM live_word_counts ORDER BY count DESC").show()

+---------+-----+
|     word|count|
+---------+-----+
|    hello|    1|
|streaming|    1|
|       is|    1|
|    world|    1|
|  awesome|    1|
+---------+-----+



### Exercise 4: The Real-Time Test (Terminal + Jupyter)
Now for the fun part. We will produce data in the terminal and instantly query it in Jupyter.

Step 1: Start the Kafka Producer (Terminal)
In your Windows terminal, run this command to open a live text prompt hooked directly to Kafka:

Bash
docker exec -it kafka kafka-console-producer --topic live_events --bootstrap-server localhost:9092
You will see a > prompt appear. Type a few sentences, hitting Enter after each one:

Plaintext
> hello world
> hello spark
> streaming is awesome
> hello streaming

#### Step 2: Query the Live Data (Jupyter)
Leave the terminal open. Go back to Jupyter and run a standard SQL query against the memory sink we created.

In [9]:
spark.sql("SELECT * FROM live_word_counts ORDER BY count DESC").show()

+---------+-----+
|     word|count|
+---------+-----+
|    hello|    2|
|streaming|    2|
|       is|    1|
|    world|    1|
|  awesome|    1|
+---------+-----+



In [10]:
# Stop the background streaming query
streaming_query.stop()

# Stop the Spark session
spark.stop()
print("Streaming stopped and resources released.")

Streaming stopped and resources released.
